# Building an AI Coding Harness from Scratch

This notebook documents the progression from a basic LLM call to a working AI coding harness.

The path I have implemented:

```text
LLM
 ↓
Tool Call
 ↓
Coding Harness
 ↓
Tool Execution
 ↓
Tool Result
 ↓
LLM
```

The implementation is without using an existing agent framework.

## 1. LLM as the Decision Maker

LLM is able to generate text but it cannot directly read files, modify a codebase or execute commands.

A coding harness gives the LLM controlled access to these through tools.

The LLM decides **what should happen**.  
The harness controls **how it actually happens**.

Basically it provides the infrastructure that the LLM needs

## 2. Connecting to Gemini

Gemini is used as the initial model provider while building the harness.

The provider is kept separate so that the model can be replaced later.

In [1]:
from google import genai
from dotenv import load_dotenv
import os

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")

In [2]:
response = client.models.generate_content(
    model=MODEL,
    contents="What is 17 + 25?"
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


17 + 25 = 42


To test the key and show that it is capable of generating text^

## 3. Understanding Function Calling

Next step is giving the model a Python function as a tool.

The model does not execute the Python function. It produces a structured function call.

In [3]:
def add_numbers(a: int, b: int) -> int:
    """Adds two numbers together and returns their sum."""
    return a + b

In [4]:
from google.genai import types

response = client.models.generate_content(
    model=MODEL,
    contents="What is 17 + 25?",
    config=types.GenerateContentConfig(
        tools=[add_numbers]
    )
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='17 + 25 = 42',
        thought_signature=b'\x12^\n\\\x01i\x14}\x13\xbcE[)\x0bx\x1a\xf5\xbfq\r\r\xef!`f.Q\x90\x08\x97\x1e\x9f"\xe6|\x03O\xc0t\xb1Ud\x9b\x930\xb0\xd8/\x97\xd8\x97\x88\x9e\xe3\x05\xb45W\xc3\x11b\xaa\xc7+%\x9d&ht\x847#\xe71\xad\x02\x95Q\n(\tjA\x00\xa3d\xa5\xa4X\xcf\xf4\xdf\xcaK\x88M'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='cEuwapPKCt6kg8UPxa3BqAU' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  prompt_token_count=110,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=110
    ),
  ],
  total_token_count=120
) model_status=None automatic_function_calling_history=[UserContent(
  parts=[
    Part(
     

### Automatic function calling

The Gemini SDK can automatically execute Python functions supplied as tools.

Automatic function calling is by default enabled for gemini, but want to implement the function calling by scratch so that gemini can be easily replaced by other models and to better understand how tools are called. Hence disabled it from here to implement it manually

```text
LLM decision
 ↓
Function call
 ↓
Validation
 ↓
Execution
 ↓
Tool result
 ↓
LLM
```

In [5]:
config = types.GenerateContentConfig(
    tools=[add_numbers],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)

response = client.models.generate_content(
    model=MODEL,
    contents="What is 17 + 25?",
    config=config,
)

print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'a': 17,
            'b': 25
          },
          id='call_955053',
          name='add_numbers'
        ),
        thought_signature=b'\x12^\n\\\x01i\x14}\x13\xd9sv\xc0\xd1\xeewq\x8d\xe3BU\x92\xf9>\xa3\xa6ks\xed\xf6\x91\xa7\x1f\x8d\x83\xc4>\xc3\xe3\xad\xd2\n&\x8b\xbaS\xf9\xf2O5B\x99\xcb\xcc\xf2r_\xce\xbd\x11\xa4hZ,\x11~/\\\x9d\xdb^\x12\x04R\xc3\x80\xbd\xbbH\x07>\xa9#t\x0fZ\t\xf0\xc1\xe5R\x93\xe0\xc5&]'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='00uwatS2B7elqfkP64CByQU' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=20,
  prompt_token_count=77,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT:

## 4. Inspecting the Function Call

A model response contains candidates, content, and parts.

A function call is stored inside a response part with information such as:

- tool name
- arguments
- function-call ID

In [6]:
candidate = response.candidates[0]
content = candidate.content

for part in content.parts:
    print(part)
    if part.function_call:
        print("Name:", part.function_call.name)
        print("Arguments:", part.function_call.args)
        print("ID:", part.function_call.id)

media_resolution=None code_execution_result=None executable_code=None file_data=None function_call=FunctionCall(
  args={
    'a': 17,
    'b': 25
  },
  id='call_955053',
  name='add_numbers'
) function_response=None inline_data=None text=None thought=None thought_signature=b'\x12^\n\\\x01i\x14}\x13\xd9sv\xc0\xd1\xeewq\x8d\xe3BU\x92\xf9>\xa3\xa6ks\xed\xf6\x91\xa7\x1f\x8d\x83\xc4>\xc3\xe3\xad\xd2\n&\x8b\xbaS\xf9\xf2O5B\x99\xcb\xcc\xf2r_\xce\xbd\x11\xa4hZ,\x11~/\\\x9d\xdb^\x12\x04R\xc3\x80\xbd\xbbH\x07>\xa9#t\x0fZ\t\xf0\xc1\xe5R\x93\xe0\xc5&]' video_metadata=None tool_call=None tool_response=None part_metadata=None audio_transcription=None media_processing=None
Name: add_numbers
Arguments: {'b': 25, 'a': 17}
ID: call_955053


Now the model can only request for the action.
It has not executed the Python function.
The harness must execute it.

In [7]:
function_call = next(
    part.function_call
    for part in content.parts
    if part.function_call
)

result = add_numbers(**function_call.args)

print("Tool result:", result)

Tool result: 42


## 5. Returning the Tool Result

After executing the tool, the result is converted into a function-response message and added to the conversation.

The model can then use that result to continue to give us response

In [8]:
tool_response = types.Part.from_function_response(
    name=function_call.name,
    response={"result": result},
)

contents = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(
                text="What is 17 + 25?"
            )
        ],
    ),
    content,
    types.Content(
        role="user",
        parts=[tool_response],
    ),
]

final_response = client.models.generate_content(
    model=MODEL,
    contents=contents,
    config=config,
)

print(final_response.text)

17 + 25 = 42


The basic interaction is now:

```text
User
 ↓
LLM
 ↓
Function Call
 ↓
Python Function
 ↓
Function Response
 ↓
LLM
 ↓
Final Response
```

This is the foundation of the coding harness.

## 6. The Harness Loop

But a coding task usually requires more than one tool call.

For example:

```text
Inspect files
 ↓
Read relevant file
 ↓
Edit file
 ↓
Run code
 ↓
Observe error
 ↓
Fix code
 ↓
Run again
```

The harness therefore needs a loop.

## 7. Multiple Tools

Creating one more tool to show mutiple as harnesses should be able to execute more than 1

For example:

In [9]:
def multiply_numbers(a: int, b: int) -> int:
    """Multiplies two numbers together and returns their product."""
    return a * b


tools = [
    add_numbers,
    multiply_numbers,
]

For now a dictionary can map tool names to Python functions

## 8. Tool Abstraction

A Tool wraps a Python function with the metadata and behavior needed by the harness.

A tool contains:

- name
- description
- Python function
- argument information
- validation
- provider-facing schema

In [10]:
from dataclasses import dataclass
from typing import Callable, Any, get_type_hints
import inspect


@dataclass
class Tool:
    name: str
    description: str
    function: Callable[..., Any]

    def call(self, arguments: dict):
        self.validate_arguments(arguments)
        return self.function(**arguments)

    def execute(self, **kwargs):
        return self.call(kwargs)

The important boundary is:

```text
LLM-generated arguments
        ↓
      Tool
        ↓
   validation
        ↓
 Python function
```

The underlying Python function should not have to trust raw model output.

## 9. Tool Registry

The harness needs a central place to register, retrieve, list, and execute tools.

So the ToolRegistry class provides one interface to make this easier

In [11]:
class ToolRegistry:

    def __init__(self):
        self.tools = {}

    def register(self, tool: Tool):
        self.tools[tool.name] = tool

    def get(self, name: str) -> Tool:
        if name not in self.tools:
            raise KeyError(f"Unknown tool: {name}")
        return self.tools[name]

    def list_tools(self):
        return list(self.tools.values())

    def execute(self, name: str, **kwargs):
        try:
            tool = self.get(name)
            result = tool.execute(**kwargs)

            return {
                "success": True,
                "tool": name,
                "result": result,
            }

        except Exception as e:
            return {
                "success": False,
                "tool": name,
                "error": type(e).__name__,
                "message": str(e),
            }

The registry creates a clean runtime boundary:

```text
Tool name
   ↓
ToolRegistry
   ↓
Tool
   ↓
Python function
```

## 10. Creating a Coding Workspace

Creating a safe workspace to give to the agent to make changes in.

In [12]:
from pathlib import Path

WORKSPACE = Path.cwd().parent / "workspace"
WORKSPACE.mkdir(parents=True, exist_ok=True)


def resolve_path(path: str) -> Path:
    target = (WORKSPACE / path).resolve()

    if not target.is_relative_to(WORKSPACE.resolve()):
        raise ValueError("Path is outside the workspace.")

    return target

The important safety property is:

```text
Requested path
      ↓
Resolved path
      ↓
Must remain inside workspace
```

This prevents basic path traversal such as `../../some_file`.

## 11. File System Tools

### List files

In [13]:
def list_files(path: str = ".") -> str:
    directory = resolve_path(path)

    if not directory.exists():
        raise FileNotFoundError(f"Directory not found: {path}")

    if not directory.is_dir():
        raise NotADirectoryError(f"Not a directory: {path}")

    files = []

    for item in sorted(directory.iterdir()):
        if item.is_dir():
            files.append(f"[DIR]  {item.name}")
        else:
            files.append(f"[FILE] {item.name}")

    return "\n".join(files) if files else "Directory is empty."

### Read and write files

In [14]:
def read_file(path: str) -> str:
    file_path = resolve_path(path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if not file_path.is_file():
        raise IsADirectoryError(f"Not a file: {path}")

    return file_path.read_text(encoding="utf-8")


def write_file(path: str, content: str) -> str:
    file_path = resolve_path(path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    file_path.write_text(content, encoding="utf-8")

    return f"Successfully wrote {path}"

### Edit files

In [15]:
def edit_file(path: str, old_text: str, new_text: str) -> str:
    file_path = resolve_path(path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    content = file_path.read_text(encoding="utf-8")
    occurrences = content.count(old_text)

    if occurrences == 0:
        raise ValueError(
            f"Could not find the specified text in {path}."
        )

    if occurrences > 1:
        raise ValueError(
            f"The specified text appears {occurrences} times in {path}. "
            "Provide more specific text."
        )

    file_path.write_text(
        content.replace(old_text, new_text),
        encoding="utf-8",
    )

    return f"Successfully edited {path}"

### Search files

In [16]:
def search_files(query: str) -> str:
    results = []

    for file_path in WORKSPACE.rglob("*"):
        if not file_path.is_file():
            continue

        try:
            content = file_path.read_text(encoding="utf-8")
        except (UnicodeDecodeError, PermissionError):
            continue

        for line_number, line in enumerate(
            content.splitlines(),
            start=1,
        ):
            if query.lower() in line.lower():
                relative_path = file_path.relative_to(WORKSPACE)

                results.append(
                    f"{relative_path}:{line_number}: {line.strip()}"
                )

    return (
        "\n".join(results)
        if results
        else f"No matches found for: {query}"
    )

These tools turn the LLM from a text generator into something that can inspect and modify a real codebase.

```text
list_files   → inspect
read_file    → inspect
search_files → locate
write_file   → create
edit_file    → modify
```

## 12. Shell Execution

Only reading and editing code is not enough.
A coding harness must also be able to run the code and see the result.

In [17]:
import subprocess


def run_command(command: str) -> str:
    result = subprocess.run(
        command,
        shell=True,
        cwd=WORKSPACE,
        capture_output=True,
        text=True,
        timeout=30,
    )

    output = []

    if result.stdout:
        output.append(f"STDOUT:\n{result.stdout}")

    if result.stderr:
        output.append(f"STDERR:\n{result.stderr}")

    output.append(f"EXIT CODE: {result.returncode}")

    return "\n".join(output)

The shell tool is simple.
The implementation adds an allowlist of commands, timeout handling, and output limits.


## 13. Registering Coding Tools

The coding tools can now be exposed through the same ToolRegistry used earlier.

In [18]:
registry = ToolRegistry()

registry.register(
    Tool(
        name="list_files",
        description="Lists files and directories inside the workspace.",
        function=list_files,
    )
)

registry.register(
    Tool(
        name="read_file",
        description="Reads a file inside the workspace.",
        function=read_file,
    )
)

registry.register(
    Tool(
        name="write_file",
        description="Creates or overwrites a file in the workspace.",
        function=write_file,
    )
)

registry.register(
    Tool(
        name="edit_file",
        description="Replaces one unique piece of text in a file.",
        function=edit_file,
    )
)

registry.register(
    Tool(
        name="search_files",
        description="Searches for text inside workspace files.",
        function=search_files,
    )
)

registry.register(
    Tool(
        name="run_command",
        description="Runs an allowed development command in the workspace.",
        function=run_command,
    )
)

print([tool.name for tool in registry.list_tools()])

['list_files', 'read_file', 'write_file', 'edit_file', 'search_files', 'run_command']


## 14. Context Management

The harness needs to preserve what happened during the current task.

The context contains:

- user messages
- model responses
- function calls
- tool results

Without this history, the LLM would not know what happened in previous iterations.

In [19]:
class ContextManager:

    def __init__(self):
        self.contents = []

    def add_user_message(self, text):
        self.contents.append(
            types.Content(
                role="user",
                parts=[types.Part.from_text(text=text)],
            )
        )

    def add_model_message(self, content):
        self.contents.append(content)

    def get_contents(self):
        return self.contents

    def clear(self):
        self.contents.clear()

One important implementation detail and constant bug I discovered while building the harness:

Each task needs fresh context.

The context should persist across iterations of one task, but it should not accidentally carry over from a previous task.

Otherwise the next run can end with invalid conversation state or contain unrelated history.

## 15. Tool Argument Validation

An LLM generates tool arguments, but model-generated data should not be passed directly into application code.

The harness validates arguments before execution.

```text
LLM
 ↓
Function Call
 ↓
Argument Validation
 ↓
Tool Execution
 ↓
Tool Result
 ↓
LLM
```


In [20]:
import inspect
from typing import get_type_hints


def validate_arguments(function, arguments: dict):
    signature = inspect.signature(function)
    type_hints = get_type_hints(function)

    for name in arguments:
        if name not in signature.parameters:
            raise TypeError(f"Unexpected argument: {name}")

    for name, parameter in signature.parameters.items():
        if (
            parameter.default is inspect.Parameter.empty
            and name not in arguments
        ):
            raise ValueError(f"Missing required argument: {name}")

    for name, value in arguments.items():
        expected_type = type_hints.get(name)

        if expected_type is None:
            continue

        if not isinstance(value, expected_type):
            raise TypeError(
                f"Argument '{name}' must be "
                f"{expected_type.__name__}, "
                f"got {type(value).__name__}"
            )

## 16. Reliability and Error Handling

Tools will fail.

Examples:

- a requested file does not exist
- an argument has the wrong type
- a command exits with an error
- a command times out
- a tool name is invalid

The harness should turn these failures into structured results rather than crashing immediately.

This allows the LLM to observe the failure and decide what to do next.

```text
Tool failure
     ↓
Structured error
     ↓
Context
     ↓
LLM
     ↓
Recovery
```

## 17. Verification

A key property of a coding harness is that it should be able to verify work.

For example:

```text
Write file
    ↓
Run file
    ↓
Observe output
    ↓
Decide whether another change is needed
```

## 18. The Complete Coding Harness Loop

All the components can now be combined:

```text
                     User Task
                         │
                         ▼
                  ┌─────────────┐
                  │   Context   │
                  └──────┬──────┘
                         │
                         ▼
                        LLM
                         │
                    Tool Call
                         │
                         ▼
                ┌────────────────┐
                │ Coding Harness │
                │                │
                │ Validate       │
                │ Registry       │
                │ Execute        │
                │ Safety         │
                └───────┬────────┘
                        │
              ┌─────────┴─────────┐
              ▼                   ▼
         Filesystem             Shell
              │                   │
              └─────────┬─────────┘
                        ▼
                   Tool Result
                        │
                        ▼
                     Context
                        │
                        ▼
                       LLM
                        │
                        ▼
                  Final Response
```

## 19. Evaluation

The model's final response is not proof that a coding task succeeded.

The harness therefore needs evaluations based on the actual workspace state.

For example:

```text
Task:
Create eval_test.py containing print("evaluation works")

Evaluation:
Does eval_test.py exist?
Does it contain the expected content?
```

This gives us:

```text
Task
 ↓
Harness
 ↓
Workspace
 ↓
Evaluation
 ↓
Pass / Fail
```

The project includes a small evaluation framework in `evaluator.py` and evaluation cases under `tests/`.

## 20. Provider Abstraction

The core harness should not depend on Gemini-specific implementation details everywhere.

The provider layer separates model communication from the rest of the harness:

```text
Coding Harness
      ↓
  LLMProvider
      ↓
GeminiProvider
      ↓
   Gemini API
```

This makes it possible to replace the underlying model provider without rewriting the entire harness.

## 21. Final Architecture

The completed implementation is organized around these components:

- **LLM Provider** — communicates with the model
- **Harness Loop** — coordinates model decisions and execution
- **Context Manager** — maintains task state
- **Tool Registry** — manages available tools
- **Tool Abstraction** — validates and executes tools
- **Filesystem Tools** — interact with the codebase
- **Shell Tool** — executes development commands
- **Safety Layer** — restricts operations
- **Evaluator** — verifies actual outcomes


## Conclusion

The core of an AI coding harness is not a complicated planning algorithm.

It is the controlled loop between a model and a real coding environment:

```text
LLM
 ↓
Tool Call
 ↓
Harness
 ↓
Execution
 ↓
Observation
 ↓
LLM
```

Everything else — tools, context, validation, safety, error handling, and evaluation — exists to make this loop reliable.

The production implementation of this notebook lives in `src/agent_learning/`.

Run using: uv run python -m agent_learning.main